# UDE Deep DiveTrain a Universal Differential Equation (neural ODE) on multiomics data to learn unknown dynamics.

In [ ]:
import syssys.path.insert(0, '..')import numpy as npimport matplotlib.pyplot as pltimport torchfrom src.synthetic_data import generate_cohortfrom src.ude_pipeline import UDENet, train_udefrom src.reynolds_ode import REYNOLDS_PARAMS

## Generate training cohortCreate a 15-patient cohort for UDE training.

In [ ]:
print('Generating 15-patient training cohort...')cohort = generate_cohort(N_resolution=5, N_chronic=5, N_death=5, seed=42, verbose=False)print(f'Generated {len(cohort)} patients')print(f'\nCohort composition:')for outcome in ['resolution', 'chronic', 'death']:    count = sum(1 for p in cohort if p['outcome'] == outcome)    print(f'  {outcome}: {count}')

## Initialize and train UDETrain a neural network to learn the unknown pathogen clearance function.

In [ ]:
print('Initializing UDE network...')nn_model = UDENet(input_dim=3, hidden_dim=16, output_dim=1)config = {    'n_epochs': 50,    'lr': 1e-3,    'hidden_dim': 16,    'batch_size': 4,    'random_seed': 42}print('\nTraining UDE...')try:    trained_model, history = train_ude(        cohort,        nn_model,        REYNOLDS_PARAMS,        config=config,        verbose=True    )    print('\nUDE training complete!')    if 'error' in history:        print(f'Note: {history["error"]}')except Exception as e:    print(f'UDE training error: {e}')    trained_model = nn_model    history = {'error': str(e)}

## Training loss curveVisualize how the neural network loss decreased during training.

In [ ]:
if 'loss' in history and len(history['loss']) > 0:    fig, ax = plt.subplots(figsize=(10, 5))    ax.plot(history['epoch'], history['loss'], linewidth=2, color='steelblue')    ax.set_xlabel('Epoch', fontsize=12)    ax.set_ylabel('Loss (MSE)', fontsize=12)    ax.set_title('UDE Training Loss Curve', fontsize=14, fontweight='bold')    ax.grid(True, alpha=0.3)    plt.tight_layout()    plt.show()        print(f'Initial loss: {history["loss"][0]:.6f}')    print(f'Final loss: {history["loss"][-1]:.6f}')    print(f'Loss reduction: {100 * (1 - history["loss"][-1] / history["loss"][0]):.1f}%')else:    print('No training history available')

## Learned clearance functionEvaluate the neural network to visualize the learned pathogen clearance dynamics.

In [ ]:
print('Evaluating learned clearance function...')try:    # Create grid of N* values    Nstar_grid = np.linspace(0, 0.8, 50)        # Fixed values for CA and f    CA_fixed = 0.2    f_fixed = 0.1        # Evaluate NN    trained_model.eval()    with torch.no_grad():        inputs = torch.tensor(            np.column_stack([Nstar_grid, np.full_like(Nstar_grid, CA_fixed), np.full_like(Nstar_grid, f_fixed)]),            dtype=torch.float32        )        outputs = trained_model(inputs).numpy().flatten()        # Plot    fig, ax = plt.subplots(figsize=(10, 6))    ax.plot(Nstar_grid, outputs, linewidth=2.5, color='darkblue')    ax.fill_between(Nstar_grid, outputs, alpha=0.3, color='steelblue')    ax.set_xlabel('N* (Early Pro-inflammatory Mediator)', fontsize=12)    ax.set_ylabel('Learned Clearance Rate', fontsize=12)    ax.set_title(f'Neural Network Learned Function (CA={CA_fixed}, f={f_fixed})', fontsize=14, fontweight='bold')    ax.grid(True, alpha=0.3)    plt.tight_layout()    plt.show()        print(f'Min clearance rate: {np.min(outputs):.6f}')    print(f'Max clearance rate: {np.max(outputs):.6f}')except Exception as e:    print(f'Evaluation error: {e}')

## SINDy symbolic regression (optional)If pysindy is available, recover interpretable equations.

In [ ]:
try:    import pysindy as ps    print('pysindy is available for symbolic regression')    print('\nNote: Full SINDy integration requires additional setup.')    print('See the full experiment pipeline for complete SINDy results.')except ImportError:    print('pysindy not installed. Skipping symbolic regression step.')    print('Install with: pip install pysindy')

## SummaryYou've trained a Universal Differential Equation that:1. Learns unknown dynamics from multiomics data2. Combines known biological equations with learned neural network terms3. Can generalize to new patients and conditionsNext steps:- See `03-results-analysis.ipynb` for comparison with MOTIF- See `04-methodology.ipynb` for technical details